In [0]:
# Bronze: Synthetic Customer Churn Data Generation
import numpy as np
import pandas as pd

np.random.seed(42)
n_customers = 5000

# Core attributes
tenure_months = np.random.gamma(shape=2, scale=15, size=n_customers).astype(int).clip(1, 72)
contract_type = np.random.choice(['Month-to-Month', 'One Year', 'Two Year'], size=n_customers, p=[0.55, 0.25, 0.20])
monthly_charges = np.round(np.random.normal(65, 25, n_customers).clip(20, 150), 2)
support_tickets = np.random.poisson(lam=1.5, size=n_customers)
payment_method = np.random.choice(['Credit Card', 'Bank Transfer', 'Electronic Check', 'Mailed Check'], size=n_customers)
internet_service = np.random.choice(['DSL', 'Fiber Optic', 'No'], size=n_customers, p=[0.35, 0.45, 0.20])
has_streaming = np.random.choice(['Yes', 'No'], size=n_customers, p=[0.5, 0.5])

# Churn probability logic (higher risk: month-to-month, high tickets, short tenure, electronic check)
churn_score = (
    (contract_type == 'Month-to-Month').astype(int) * 0.35
    + (support_tickets >= 3).astype(int) * 0.25
    + (tenure_months < 6).astype(int) * 0.20
    + (payment_method == 'Electronic Check').astype(int) * 0.15
    + np.random.normal(0, 0.15, n_customers)
)
churn = (churn_score > 0.45).astype(int)

df_bronze = pd.DataFrame({
    'customer_id': [f'CUST-{i:05d}' for i in range(n_customers)],
    'tenure_months': tenure_months,
    'contract_type': contract_type,
    'monthly_charges': monthly_charges,
    'support_tickets': support_tickets,
    'payment_method': payment_method,
    'internet_service': internet_service,
    'has_streaming': has_streaming,
    'churned': churn
})

print(f"Total customers: {len(df_bronze)}")
print(f"Churn rate: {df_bronze['churned'].mean():.1%}")
display(df_bronze.head(10))

Total customers: 5000
Churn rate: 28.6%


customer_id,tenure_months,contract_type,monthly_charges,support_tickets,payment_method,internet_service,has_streaming,churned
CUST-00000,35,Two Year,24.72,2,Mailed Check,DSL,No,0
CUST-00001,22,Month-to-Month,42.0,3,Electronic Check,DSL,Yes,1
CUST-00002,20,Month-to-Month,61.3,0,Mailed Check,Fiber Optic,Yes,0
CUST-00003,20,One Year,42.83,1,Credit Card,DSL,No,0
CUST-00004,69,Month-to-Month,44.61,2,Mailed Check,Fiber Optic,No,1
CUST-00005,43,Month-to-Month,56.25,4,Bank Transfer,Fiber Optic,No,1
CUST-00006,16,Month-to-Month,81.24,2,Bank Transfer,Fiber Optic,No,0
CUST-00007,37,Month-to-Month,56.43,1,Bank Transfer,DSL,Yes,1
CUST-00008,29,Month-to-Month,91.96,1,Credit Card,DSL,Yes,0
CUST-00009,3,Two Year,40.06,2,Mailed Check,Fiber Optic,No,0


In [0]:
# Write Bronze table to Delta
spark.createDataFrame(df_bronze).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_customer_churn")

print(f"Bronze table written: {spark.table('bronze_customer_churn').count()} records")

Bronze table written: 5000 records
